In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from supabase import create_client, Client

load_dotenv(find_dotenv(), override=True)

supabase: Client = create_client(
    os.environ.get("SUPABASE_URL", ""), os.environ.get("SUPABASE_KEY", "")
)

In [3]:
from sentence_transformers import SentenceTransformer

model_id = "Qwen/Qwen3-Embedding-0.6B"
model = SentenceTransformer(model_id)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [5]:
query = "Could you tell me about the history of the porcelain? "
embeddings = model.encode([query], convert_to_numpy=True).tolist()

results = supabase.rpc(
    "match_paragraphs_any", {"query_embedding": embeddings[0], "match_count": 2}
).execute()

section_idxs = [item["section_idx"] for item in results.data]
sections = (
    supabase.table("sections")
    .select("en_text", "page_idx")
    .in_("section_idx", section_idxs)
    .execute()
).data

images = (
    supabase.table("images")
    .select("image_data")
    .in_("page_idx", [item["page_idx"] for item in sections])
    .execute()
).data

sections, images

([{'en_text': 'The Chinese term ‘ci’ (translated as porcelain in English) refers to all ceramics that are fired at high temperatures, including porcelain and stoneware.\n\nIn the West, the term porcelain refers specifically to white ceramics made with a special type of clay called kaolin and fired to a temperature of about 1300◦C, which results in a translucent, glassy material that makes a ringing sound when struck.\n\nStoneware is used to refer to related ceramics that are similarly hard and dense, but which are made with grey or brown clay, may or may not be white-bodied, do not transmit light, and are fired to a slightly lower temperature of 1000 to 1250◦C.\n\nCeramics fired below this temperature range are called earthenwares.\n\nThe terms ‘proto-porcelain’ or ‘porcellaneous’ are sometimes used to describe early ceramics made with some of the same ingredients and physical characteristics of porcelain.',
   'page_idx': 5},
  {'en_text': 'The word porcelain originated with the Venet

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 配置 Reranker
rerank_model_path = "Qwen/Qwen3-Reranker-0.6B"  # 或您的本地路径
rerank_tokenizer = AutoTokenizer.from_pretrained(rerank_model_path, padding_side="left")
rerank_model = AutoModelForCausalLM.from_pretrained(rerank_model_path).eval()

# Reranker 常量设置
TOKEN_FALSE_ID = rerank_tokenizer.convert_tokens_to_ids("no")
TOKEN_TRUE_ID = rerank_tokenizer.convert_tokens_to_ids("yes")
MAX_LENGTH = 8192
PREFIX = '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
PREFIX_TOKENS = rerank_tokenizer.encode(PREFIX, add_special_tokens=False)
SUFFIX_TOKENS = rerank_tokenizer.encode(SUFFIX, add_special_tokens=False)
TASK_INSTRUCTION = (
    "Given a museum gallery query, retrieve relevant passages that answer the query"
)


def rerank_documents(query: str, documents: list[str]) -> list[str]:
    """使用 Qwen3-Reranker 对文档进行重排序"""
    if not documents:
        return []

    # 构建输入对
    pairs = [
        f"<Instruct>: {TASK_INSTRUCTION}\n<Query>: {query}\n<Document>: {doc}"
        for doc in documents
    ]

    # 预处理输入
    inputs = rerank_tokenizer(
        pairs,
        padding=False,
        truncation="longest_first",
        return_attention_mask=False,
        max_length=MAX_LENGTH - len(PREFIX_TOKENS) - len(SUFFIX_TOKENS),
    )

    for i, ele in enumerate(inputs["input_ids"]):
        inputs["input_ids"][i] = PREFIX_TOKENS + ele + SUFFIX_TOKENS

    inputs = rerank_tokenizer.pad(inputs, padding=True, return_tensors="pt")

    # 计算分值
    with torch.no_grad():
        logits = rerank_model(**inputs).logits[:, -1, :]
        true_vector = logits[:, TOKEN_TRUE_ID]
        false_vector = logits[:, TOKEN_FALSE_ID]
        batch_scores = torch.stack([false_vector, true_vector], dim=1)
        batch_scores = torch.nn.functional.log_softmax(batch_scores, dim=1)
        scores = batch_scores[:, 1].exp().tolist()

    # 结合分值排序
    scored_docs = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)
    # 返回评分高于阈值的文档（例如 0.5），或返回前 N 个
    return [doc for score, doc in scored_docs if score > 0.3]

d:\HKU\Inno Wing RA\UBC Exchange\museum_tour_guide\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
